# 01 — Carga da RAIS Vínculos no DuckDB (2023–2025)

Cria um banco `rais_<ano>.duckdb` por ano a partir dos **microdados públicos da RAIS Vínculos** (MTE/PDET), com as colunas extras `ano`, `regiao_arquivo`, `uf` e `regiao_geografica`, e uma view `rais_vinc_brasil` que une as regiões do ano. Esses bancos são a entrada do notebook `02_analise_rais.ipynb`.

## Antes de executar
1. Baixe os arquivos **RAIS Vínculos Públicos** de cada ano em `ftp://ftp.mtps.gov.br/pdet/microdados/` (pasta `RAIS/<ano>/`), um arquivo compactado por região: `CENTRO_OESTE`, `MG_ES_RJ`, `NI`, `NORDESTE`, `NORTE`, `SP` e `SUL`.
2. Descompacte de modo que cada ano fique assim (o `.COMT` é o arquivo de dados em texto, separado por vírgula, codificação latin-1):
```
<DADOS_PATH>/<ano>/RAIS_VINC_PUB_<REGIAO>/RAIS_VINC_PUB_<REGIAO>.COMT
```
3. Defina `DADOS_PATH` (variável de ambiente `RAIS_DADOS_PATH`; o padrão é uma pasta do Google Drive).

## Observações metodológicas
- Todas as colunas são lidas como texto (`all_varchar`), preservando códigos com zeros à esquerda.
- **UF e região** são derivadas dos dois primeiros dígitos do **código do município do estabelecimento** (tabela IBGE de UFs). Não representam a residência do trabalhador. Municípios não identificados (arquivo `NI`) ficam com UF e região `NI`.
- **Memória:** os anos são processados um por vez; ao fim de cada ano a conexão é fechada e a memória liberada. O DuckDB respeita `MEMORY_LIMIT` e usa disco temporário (`TEMP_DIR`) quando necessário. Se o Colab ainda esgotar a RAM, execute um ano por vez (`ANOS = [2025]`, por exemplo) e reinicie a sessão entre eles.
- Os bancos não são redistribuídos neste repositório (são grandes e os microdados já são públicos no site do MTE).

In [ ]:
import sys, os, gc
EM_COLAB = 'google.colab' in sys.modules
if EM_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
try:
    import duckdb
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])
    import duckdb
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
# Parâmetros
ANOS = [2023, 2024, 2025]
DADOS_PATH = os.environ.get('RAIS_DADOS_PATH', '/content/drive/MyDrive/MBA_USP_BIA/TCC/CAGED')
REGIOES = ['CENTRO_OESTE', 'MG_ES_RJ', 'NI', 'NORDESTE', 'NORTE', 'SP', 'SUL']

# Memória: limite do DuckDB (deixe ~2 GB livres para o Python/Colab; ajuste ao seu plano) e pasta de
# arquivos temporários em disco local (o DuckDB descarrega para lá o que não cabe na memória)
MEMORY_LIMIT = os.environ.get('RAIS_MEMORY_LIMIT', '8GB')
TEMP_DIR = os.environ.get('RAIS_TEMP_DIR', '/tmp/duckdb_tmp')
os.makedirs(TEMP_DIR, exist_ok=True)

pasta_ano = lambda ano: f'{DADOS_PATH}/{ano}'
db_path   = lambda ano: f'{DADOS_PATH}/{ano}/rais_{ano}.duckdb'
arquivo   = lambda ano, reg: f'{DADOS_PATH}/{ano}/RAIS_VINC_PUB_{reg}/RAIS_VINC_PUB_{reg}.COMT'

print('DADOS_PATH:', DADOS_PATH)
for ano in ANOS:
    faltando = [r for r in REGIOES if not os.path.exists(arquivo(ano, r))]
    print(ano, '| arquivos encontrados:', len(REGIOES) - len(faltando), '/', len(REGIOES),
          '| faltando:', faltando or '—')

In [ ]:
# Tabelas auxiliares: código IBGE da UF (2 primeiros dígitos do município) -> UF e região
UF_POR_COD = {
 '11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
 '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL','28':'SE','29':'BA',
 '31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR','42':'SC','43':'RS',
 '50':'MS','51':'MT','52':'GO','53':'DF'}
REGIAO_POR_COD = {c: ('Norte' if c[0]=='1' else 'Nordeste' if c[0]=='2' else 'Sudeste' if c[0]=='3'
                      else 'Sul' if c[0]=='4' else 'Centro-Oeste') for c in UF_POR_COD}
def case_sql(mapa):
    whens = ' '.join(f"WHEN '{k}' THEN '{v}'" for k, v in mapa.items())
    return f"""CASE SUBSTR("Município - Código", 1, 2) {whens} ELSE 'NI' END"""
CASE_UF, CASE_REGIAO = case_sql(UF_POR_COD), case_sql(REGIAO_POR_COD)
assert REGIAO_POR_COD['35'] == 'Sudeste' and REGIAO_POR_COD['53'] == 'Centro-Oeste' and REGIAO_POR_COD['11'] == 'Norte'

In [ ]:
def carregar_ano(ano):
    """Cria/atualiza rais_<ano>.duckdb com uma tabela por região e a view rais_vinc_brasil."""
    if not any(os.path.exists(arquivo(ano, r)) for r in REGIOES):
        print(f'[AVISO] {ano}: nenhum arquivo de dados encontrado em {pasta_ano(ano)}; ano ignorado.'); return False
    con = duckdb.connect(db_path(ano))
    con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
    con.execute(f"SET temp_directory='{TEMP_DIR}'")
    con.execute('SET preserve_insertion_order=false')   # reduz o uso de memória na leitura dos CSVs
    try:
        for reg in REGIOES:
            arq = arquivo(ano, reg)
            if not os.path.exists(arq):
                print(f'[AVISO] {ano}: não encontrado, pulando: {arq}'); continue
            tabela = f'rais_vinc_{reg.lower()}'
            con.execute(f"""
                CREATE OR REPLACE TABLE {tabela} AS
                SELECT *, {ano} AS ano, '{reg.lower()}' AS regiao_arquivo,
                       {CASE_UF} AS uf, {CASE_REGIAO} AS regiao_geografica
                FROM read_csv('{arq}', delim=',', header=true, quote='"', encoding='latin-1',
                              sample_size=10000, all_varchar=true)""")
            n = con.execute(f'SELECT COUNT(*) FROM {tabela}').fetchone()[0]
            print(f'  {ano} {tabela}: {n:,} linhas')
        tabelas = con.execute("""SELECT table_name FROM information_schema.tables
                                 WHERE table_name LIKE 'rais_vinc_%' AND table_type = 'BASE TABLE'
                                 ORDER BY table_name""").df()['table_name'].tolist()
        assert tabelas, f'Nenhuma tabela criada para {ano}'
        con.execute('CREATE OR REPLACE VIEW rais_vinc_brasil AS ' + ' UNION ALL '.join(f'SELECT * FROM {t}' for t in tabelas))
        total = con.execute('SELECT COUNT(*) FROM rais_vinc_brasil').fetchone()[0]
        print(f'✅ {ano}: view rais_vinc_brasil com {total:,} vínculos ({len(tabelas)} tabelas)')
        return True
    finally:
        con.close()          # fecha o banco e libera o buffer de memória do DuckDB
        del con
        gc.collect()

ANOS_OK = [ano for ano in ANOS if carregar_ano(ano)]

## Verificações de sanidade
Totais brutos (todos os vínculos, ativos ou não) por região e sexo, e a ocupação 2112-20 por região. O estoque oficial (ativos em 31/12) é validado no notebook `02_analise_rais.ipynb`.

In [ ]:
for ano in ANOS_OK:
    con = duckdb.connect(db_path(ano), read_only=True)
    try:
        print(f'===== {ano} =====')
        display(con.execute("""SELECT regiao_geografica, COUNT(*) AS total FROM rais_vinc_brasil
                               GROUP BY 1 ORDER BY 1""").df())
        display(con.execute("""SELECT "Sexo - Código" AS sexo, COUNT(*) AS total FROM rais_vinc_brasil
                               GROUP BY 1 ORDER BY 1""").df())
        display(con.execute("""
            SELECT regiao_geografica,
                   SUM(CASE WHEN "Sexo - Código" = '1' THEN 1 ELSE 0 END) AS masculino,
                   SUM(CASE WHEN "Sexo - Código" = '2' THEN 1 ELSE 0 END) AS feminino,
                   COUNT(*) AS total
            FROM rais_vinc_brasil WHERE "CBO 2002 Ocupação - Código" = '211220'
            GROUP BY 1 ORDER BY 1""").df())
    finally:
        con.close()